# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA (Time Series)

**Mata Kuliah:** Analisis Big Data
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)
**Nama Mahasiswa:** Wahyu Andika
**NIM:** 202310370311075
**Username GitHub:** dikameo

---

Analisis eksplorasi **PEMS-BAY** (kecepatan lalu lintas 325 sensor, resolusi 5 menit, 1 Januari s.d. 30 Juni 2017) dari data hasil cleaning Milestone 2 (`data/processed/pems_bay_clean.parquet`).

## Panduan Pengerjaan Milestone 3

1. **Sumber Data:** dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning_timeseries.ipynb`).
2. **Analisis Temporal / Windowing:** Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, serta DuckDB window function `OVER (...)` (Bagian 2).
3. **Visualisasi Interaktif:** 9 visualisasi interaktif Plotly (Bagian 3).
4. **Perumusan Insight:** insight analitik berdasarkan hasil visualisasi (Bagian 4).

## Catatan Dataset

1. Dataset ini punya dimensi waktu asli (`t0`) dan dimensi ruang (koordinat sensor), sehingga analisis temporal dan spasial berjalan langsung tanpa proxy.
2. Satu baris mewakili satu sensor pada satu waktu referensi `t0` dengan jendela 12 langkah. Kolom `x_t+0_d0` menyimpan kecepatan saat ini dan membentuk deret waktu per sensor.
3. Milestone 2 sudah mengubah kecepatan `0` (sensor gagal baca) menjadi `null`, sehingga agregasi `mean` mengabaikannya.
4. Dataset berasal dari California dan bukan dataset Indonesia. Hal ini dicatat sebagai keterbatasan terhadap spesifikasi tugas.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari `data/processed/` menggunakan Polars (lazy).

In [1]:
import json
from pathlib import Path

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'pems_bay_clean.parquet'
SENSOR_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pems-bay' / 'sensor_graph'

if not CLEAN_PATH.exists():
    raise FileNotFoundError(f'{CLEAN_PATH} belum ada. Jalankan 02_data_cleaning_timeseries.ipynb terlebih dahulu.')

lf = pl.scan_parquet(CLEAN_PATH)
n_rows = lf.select(pl.len()).collect(engine='streaming').item()
print(f'Jumlah baris data bersih: {n_rows:,}')

HARI = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

# Deret kecepatan saat ini per sensor; dipakai oleh seluruh analisis berikutnya
seri = lf.select(
    'node_id', 't0', 'split',
    pl.col('x_t+0_d0').alias('kecepatan'),
)
lf.select('node_id', 't0', 'split', 'x_t+0_d0', 'n_hilang').head(5).collect(engine='streaming')

Jumlah baris data bersih: 16,930,225


node_id,t0,split,x_t+0_d0,n_hilang
i64,datetime[μs],str,f64,i8
0,2017-05-25 18:45:00,"""test""",73.1,0
1,2017-05-25 18:45:00,"""test""",22.5,0
2,2017-05-25 18:45:00,"""test""",56.7,0
3,2017-05-25 18:45:00,"""test""",32.3,0
4,2017-05-25 18:45:00,"""test""",58.0,0


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Tahapan:
1. Rata-rata kecepatan seluruh sensor per timestamp (52 ribu titik) agar ringan untuk operasi window.
2. Polars `group_by_dynamic()`: agregasi ke interval **1 hari** (`every='1d'`).
3. Polars `rolling_mean()`: rata-rata bergerak 7 hari untuk meredam siklus mingguan.
4. Polars `shift()`: selisih kecepatan harian dengan hari sebelumnya.
5. DuckDB window function `AVG(...) OVER (...)` dan `LAG(...) OVER (...)` sebagai verifikasi silang.

In [2]:
jaringan = (
    seri.group_by('t0')
    .agg(pl.col('kecepatan').mean().alias('kecepatan_rata'), pl.col('kecepatan').count().alias('sensor_terbaca'))
    .sort('t0')
    .collect(engine='streaming')
)

harian = (
    jaringan.group_by_dynamic('t0', every='1d')
    .agg(pl.col('kecepatan_rata').mean().alias('kecepatan_harian'), pl.col('sensor_terbaca').mean().alias('sensor_terbaca'))
    .with_columns(
        pl.col('kecepatan_harian').rolling_mean(window_size=7).alias('ma_7'),
        pl.col('kecepatan_harian').shift(1).alias('kemarin'),
    )
    .with_columns((pl.col('kecepatan_harian') - pl.col('kemarin')).alias('perubahan_harian'))
    .rename({'t0': 'tanggal'})
)
harian

tanggal,kecepatan_harian,sensor_terbaca,ma_7,kemarin,perubahan_harian
datetime[μs],f64,f64,f64,f64,f64
2017-01-01 00:00:00,66.912854,325.0,null,null,null
2017-01-02 00:00:00,64.12546,325.0,null,66.912854,-2.787394
2017-01-03 00:00:00,61.677295,325.0,null,64.12546,-2.448166
2017-01-04 00:00:00,61.31488,325.0,null,61.677295,-0.362415
2017-01-05 00:00:00,62.261793,325.0,null,61.31488,0.946912
…,…,…,…,…,…
2017-06-26 00:00:00,61.587004,325.0,62.07399,65.176566,-3.589562
2017-06-27 00:00:00,60.926279,325.0,62.118086,61.587004,-0.660725
2017-06-28 00:00:00,60.619161,325.0,62.144475,60.926279,-0.307118


In [3]:
# Verifikasi silang dengan window function DuckDB
harian_duckdb = duckdb.sql('''
    SELECT
        tanggal,
        AVG(kecepatan_harian) OVER (ORDER BY tanggal ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma_7_duckdb,
        kecepatan_harian - LAG(kecepatan_harian) OVER (ORDER BY tanggal) AS perubahan_duckdb
    FROM harian
    ORDER BY tanggal
''').pl()

verifikasi = harian.join(harian_duckdb, on='tanggal')
# rolling_mean Polars bernilai null pada 6 hari pertama, DuckDB menghitung dari data yang tersedia
cocok_ma = verifikasi.drop_nulls('ma_7').select(((pl.col('ma_7') - pl.col('ma_7_duckdb')).abs() < 1e-9).all()).item()
cocok_lag = verifikasi.drop_nulls('perubahan_harian').select(((pl.col('perubahan_harian') - pl.col('perubahan_duckdb')).abs() < 1e-9).all()).item()
print('Polars rolling_mean cocok dengan DuckDB AVG OVER :', cocok_ma)
print('Polars shift cocok dengan DuckDB LAG OVER        :', cocok_lag)

Polars rolling_mean cocok dengan DuckDB AVG OVER : True
Polars shift cocok dengan DuckDB LAG OVER        : True


## Bagian 3: Visualisasi Interaktif (Plotly)

Seluruh grafik dibangun dari data teragregasi di Polars (puluhan sampai ribuan titik) agar ringan di browser, bukan dari 16,9 juta baris mentah.

## Visualisasi 1: Tren Kecepatan Harian (Line Chart dengan Range Slider)

Rata-rata kecepatan seluruh sensor per hari, dengan rata-rata bergerak 7 hari. Latar abu-abu menandai rentang `val` dan `test`.

In [4]:
bagian_split = (
    seri.group_by('split').agg(pl.col('t0').min().alias('mulai'), pl.col('t0').max().alias('akhir'))
    .sort('mulai').collect(engine='streaming')
)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=harian['tanggal'], y=harian['kecepatan_harian'], mode='lines', name='Kecepatan harian'))
fig1.add_trace(go.Scatter(x=harian['tanggal'], y=harian['ma_7'], mode='lines', name='Rata-rata bergerak 7 hari', line=dict(width=3)))
for baris in bagian_split.filter(pl.col('split') != 'train').iter_rows(named=True):
    fig1.add_vrect(x0=baris['mulai'], x1=baris['akhir'], opacity=0.08, line_width=0,
                   annotation_text=baris['split'], annotation_position='top left')
fig1.update_layout(
    title='Tren rata-rata kecepatan seluruh sensor per hari (mph)',
    xaxis=dict(title='Tanggal', rangeslider=dict(visible=True)),
    yaxis_title='Kecepatan rata-rata (mph)',
)
fig1.show()

## Visualisasi 2: Distribusi Kecepatan (Histogram)

Data dikelompokkan (binning) di Polars lebih dulu agar grafik tidak mengirim jutaan titik ke browser.

In [5]:
hist = (
    seri.drop_nulls('kecepatan')
    .select(((pl.col('kecepatan') // 2.5) * 2.5).alias('mulai'))
    .group_by('mulai').agg(pl.len().alias('jumlah')).sort('mulai')
    .collect(engine='streaming')
)

fig2 = px.bar(
    hist, x='mulai', y='jumlah',
    title='Distribusi kecepatan lalu lintas (bin 2,5 mph)',
    labels={'mulai': 'Kecepatan (mph)', 'jumlah': 'Jumlah pengamatan'},
)
fig2.show()
seri.select(pl.col('kecepatan').mean().alias('rata_rata'),
            pl.col('kecepatan').median().alias('median'), pl.col('kecepatan').std().alias('std')).collect(engine='streaming')

rata_rata,median,std
f64,f64,f64
62.619353,65.3,9.589682


## Visualisasi 3: Korelasi Kecepatan Antar Horizon Prediksi (Heatmap)

Korelasi Pearson antara kecepatan saat ini (`x_t+0`) dengan kecepatan pada beberapa langkah ke depan (`y_t+k`, k x 5 menit). Dihitung dari subset deterministik 20% sensor (`node_id % 5 == 0`) agar ringan di memori.

In [6]:
HORIZON = [1, 3, 6, 9, 12]
kolom_h = ['x_t+0_d0'] + [f'y_t+{k}_d0' for k in HORIZON]
nama_h = ['Sekarang'] + [f'+{k * 5} menit' for k in HORIZON]

subset = (
    lf.filter(pl.col('node_id') % 5 == 0)
    .select(kolom_h)
    .drop_nulls()
    .collect(engine='streaming')
)
korelasi = subset.corr()

fig3 = px.imshow(
    korelasi.to_numpy().round(3), x=nama_h, y=nama_h, text_auto=True,
    color_continuous_scale='RdBu_r', zmin=0, zmax=1,
    title=f'Korelasi kecepatan sekarang vs horizon prediksi (n={subset.height:,} baris)',
)
fig3.show()
korelasi.with_columns(pl.Series('horizon', nama_h)).select('horizon', *kolom_h)

horizon,x_t+0_d0,y_t+1_d0,y_t+3_d0,y_t+6_d0,y_t+9_d0,y_t+12_d0
str,f64,f64,f64,f64,f64,f64
"""Sekarang""",1.0,0.984058,0.939762,0.866788,0.795168,0.725178
"""+5 menit""",0.984058,1.0,0.963176,0.891025,0.818719,0.748503
"""+15 menit""",0.939762,0.963176,1.0,0.939761,0.866785,0.795165
"""+30 menit""",0.866788,0.891025,0.939761,1.0,0.93976,0.866784
"""+45 menit""",0.795168,0.818719,0.866785,0.93976,1.0,0.939759
"""+60 menit""",0.725178,0.748503,0.795165,0.866784,0.939759,1.0


## Visualisasi 4: Komposisi Kondisi Lalu Lintas per Jam (Stacked Bar)

Kecepatan dikelompokkan menjadi tiga kondisi: **macet** (< 40 mph), **padat** (40 s.d. < 60 mph), dan **lancar** (>= 60 mph). Batas ini adalah kategori deskriptif untuk analisis, bukan standar resmi.

In [7]:
kondisi = (
    seri.drop_nulls('kecepatan')
    .with_columns(
        pl.col('t0').dt.hour().alias('jam'),
        pl.when(pl.col('kecepatan') < 40).then(pl.lit('Macet (<40)'))
        .when(pl.col('kecepatan') < 60).then(pl.lit('Padat (40-60)'))
        .otherwise(pl.lit('Lancar (>=60)')).alias('kondisi'),
    )
    .group_by('jam', 'kondisi').agg(pl.len().alias('jumlah'))
    .sort('jam', 'kondisi')
    .collect(engine='streaming')
    .with_columns((pl.col('jumlah') / pl.col('jumlah').sum().over('jam') * 100).round(2).alias('persen'))
)

fig4 = px.bar(
    kondisi, x='jam', y='persen', color='kondisi', barmode='stack',
    category_orders={'kondisi': ['Macet (<40)', 'Padat (40-60)', 'Lancar (>=60)']},
    color_discrete_map={'Macet (<40)': '#d62728', 'Padat (40-60)': '#ff7f0e', 'Lancar (>=60)': '#2ca02c'},
    title='Komposisi kondisi lalu lintas per jam dalam sehari (%)',
    labels={'jam': 'Jam', 'persen': 'Persen pengamatan', 'kondisi': 'Kondisi'},
)
fig4.show()
kondisi.filter(pl.col('kondisi') == 'Macet (<40)').sort('persen', descending=True).head(5)

jam,kondisi,jumlah,persen
i8,str,u32,f64
17,"""Macet (<40)""",152281,21.57
18,"""Macet (<40)""",114657,16.24
8,"""Macet (<40)""",108348,15.35
16,"""Macet (<40)""",107809,15.27
9,"""Macet (<40)""",79934,11.32


## Visualisasi 5: Siklus Mingguan dan Harian (Heatmap Hari x Jam)

Rata-rata kecepatan menurut hari dalam seminggu dan jam. Pola berulang ini adalah komponen musiman jangka pendek pada deret waktu.

In [8]:
hari_jam = (
    seri.drop_nulls('kecepatan')
    .group_by(pl.col('t0').dt.weekday().alias('hari_no'), pl.col('t0').dt.hour().alias('jam'))
    .agg(pl.col('kecepatan').mean().round(2).alias('kecepatan_rata'))
    .sort('hari_no', 'jam')
    .collect(engine='streaming')
    .with_columns(pl.col('hari_no').map_elements(lambda n: HARI[n - 1], return_dtype=pl.Utf8).alias('hari'))
)

fig5 = px.density_heatmap(
    hari_jam, x='jam', y='hari', z='kecepatan_rata', histfunc='avg', color_continuous_scale='RdYlGn',
    category_orders={'hari': HARI},
    title='Rata-rata kecepatan menurut hari dan jam (mph)',
    labels={'jam': 'Jam', 'hari': 'Hari', 'kecepatan_rata': 'Kecepatan (mph)'},
)
fig5.show()
hari_jam.sort('kecepatan_rata').head(5)

hari_no,jam,kecepatan_rata,hari
i8,i8,f64,str
4,17,48.98,"""Kamis"""
3,17,49.2,"""Rabu"""
2,17,49.85,"""Selasa"""
2,8,50.75,"""Selasa"""
3,8,51.39,"""Rabu"""


## Visualisasi 6: Peta Kecepatan Rata-rata per Sensor (Spasial)

Setiap titik adalah satu sensor di San Francisco Bay Area. `node_id` dipetakan ke `sensor_id` lewat `adj_mx_mapping.json` (urutan daftar `sensor_ids`), lalu digabung dengan koordinat dari `sensor_locations.csv`.

In [9]:
pemetaan = json.loads((SENSOR_DIR / 'adj_mx_mapping.json').read_text(encoding='utf-8'))
sensor_ids = pemetaan['sensor_ids']
peta_node = pl.DataFrame({'node_id': list(range(len(sensor_ids))), 'sensor_id': [int(s) for s in sensor_ids]})
lokasi = pl.read_csv(SENSOR_DIR / 'sensor_locations.csv')

per_sensor = (
    seri.group_by('node_id')
    .agg(
        pl.col('kecepatan').mean().round(2).alias('kecepatan_rata'),
        (pl.col('kecepatan') < 40).mean().mul(100).round(2).alias('persen_macet'),
    )
    .collect(engine='streaming')
    .join(peta_node, on='node_id')
    .join(lokasi, on='sensor_id')
)
print(f'Sensor dengan koordinat: {per_sensor.height} dari {len(sensor_ids)}')

fig6 = px.scatter_map(
    per_sensor, lat='latitude', lon='longitude', color='kecepatan_rata', size='persen_macet',
    hover_name='sensor_id', hover_data={'kecepatan_rata': True, 'persen_macet': True, 'latitude': False, 'longitude': False},
    color_continuous_scale='RdYlGn', size_max=18, zoom=10, map_style='open-street-map',
    title='Kecepatan rata-rata per sensor (ukuran titik = persen waktu macet)',
    labels={'kecepatan_rata': 'Kecepatan (mph)', 'persen_macet': 'Persen macet'},
)
fig6.show()
per_sensor.sort('persen_macet', descending=True).head(5)

Sensor dengan koordinat: 325 dari 325


node_id,kecepatan_rata,persen_macet,sensor_id,latitude,longitude
i64,f64,f64,i64,f64,f64
26,56.0,18.44,400172,37.376264,-121.939712
82,56.63,18.42,400586,37.376835,-121.942537
121,55.76,18.19,400895,37.380958,-121.960741
134,54.77,17.36,400971,37.368669,-121.901336
295,56.67,17.14,407339,37.351571,-122.060285


## Visualisasi 7: Hari Kerja vs Akhir Pekan (Profil Harian)

Perbandingan profil kecepatan sepanjang hari (resolusi 5 menit) antara hari kerja dan akhir pekan.

In [10]:
profil = (
    jaringan.with_columns(
        (pl.col('t0').dt.weekday() >= 6).alias('akhir_pekan'),
        (pl.col('t0').dt.hour() * 60 + pl.col('t0').dt.minute()).alias('menit'),
    )
    .group_by('akhir_pekan', 'menit').agg(pl.col('kecepatan_rata').mean().round(2).alias('kecepatan_rata'))
    .sort('akhir_pekan', 'menit')
    .with_columns(pl.when(pl.col('akhir_pekan')).then(pl.lit('Akhir pekan')).otherwise(pl.lit('Hari kerja')).alias('jenis_hari'))
)

fig7 = px.line(
    profil, x='menit', y='kecepatan_rata', color='jenis_hari',
    title='Profil kecepatan sepanjang hari: hari kerja vs akhir pekan',
    labels={'menit': 'Menit sejak 00:00', 'kecepatan_rata': 'Kecepatan rata-rata (mph)', 'jenis_hari': 'Jenis hari'},
)
fig7.update_xaxes(tickvals=list(range(0, 1441, 180)), ticktext=[f'{m // 60:02d}:00' for m in range(0, 1441, 180)])
fig7.show()
profil.group_by('jenis_hari').agg(pl.col('kecepatan_rata').min().alias('minimum'), pl.col('kecepatan_rata').mean().round(2).alias('rata_rata'))

jenis_hari,minimum,rata_rata
str,f64,f64
"""Akhir pekan""",63.19,65.28
"""Hari kerja""",49.12,60.89


## Visualisasi 8: Sensor Paling Sering Macet (Bar Chart)

Lima belas sensor dengan persentase waktu macet (< 40 mph) tertinggi, diambil dari tabel per sensor pada Visualisasi 6.

In [11]:
top_macet = per_sensor.sort('persen_macet', descending=True).head(15).with_columns(pl.col('sensor_id').cast(pl.Utf8))

fig8 = px.bar(
    top_macet, x='sensor_id', y='persen_macet', color='kecepatan_rata', color_continuous_scale='RdYlGn',
    title='15 sensor dengan persentase waktu macet tertinggi',
    labels={'sensor_id': 'Sensor', 'persen_macet': 'Persen waktu macet', 'kecepatan_rata': 'Kecepatan rata-rata'},
)
fig8.update_xaxes(type='category')
fig8.show()
top_macet.select('sensor_id', 'persen_macet', 'kecepatan_rata', 'latitude', 'longitude').head(5)

sensor_id,persen_macet,kecepatan_rata,latitude,longitude
str,f64,f64,f64,f64
"""400172""",18.44,56.0,37.376264,-121.939712
"""400586""",18.42,56.63,37.376835,-121.942537
"""400895""",18.19,55.76,37.380958,-121.960741
"""400971""",17.36,54.77,37.368669,-121.901336
"""407339""",17.14,56.67,37.351571,-122.060285


## Visualisasi 9: Pergeseran Kecepatan per Bulan (Box Plot)

Distribusi kecepatan rata-rata harian jaringan per bulan, untuk melihat apakah level kecepatan bergeser antar bulan.

In [12]:
bulanan = harian.with_columns(pl.col('tanggal').dt.strftime('%Y-%m').alias('bulan'))

fig9 = px.box(
    bulanan, x='bulan', y='kecepatan_harian', points='all',
    title='Distribusi kecepatan rata-rata harian per bulan',
    labels={'bulan': 'Bulan', 'kecepatan_harian': 'Kecepatan harian (mph)'},
)
fig9.show()
bulanan.group_by('bulan').agg(
    pl.col('kecepatan_harian').mean().round(2).alias('rata_rata'),
    pl.col('kecepatan_harian').min().round(2).alias('minimum'),
).sort('bulan')

bulan,rata_rata,minimum
str,f64,f64
"""2017-01""",62.47,58.33
"""2017-02""",62.65,59.49
"""2017-03""",62.65,58.67
"""2017-04""",63.34,60.71
"""2017-05""",62.45,60.24
"""2017-06""",62.17,60.35


## Angka Pendukung Insight

Cell ini menghitung angka dari data bersih yang dipakai pada insight di bawah.

In [13]:
print('Kecepatan rata-rata jaringan:', round(jaringan['kecepatan_rata'].mean(), 2), 'mph')

print()
print('Jam dengan kecepatan rata-rata terendah dan tertinggi:')
per_jam = hari_jam.group_by('jam').agg(pl.col('kecepatan_rata').mean().round(2)).sort('kecepatan_rata')
print(per_jam.head(3))
print(per_jam.tail(3))

print()
print('Hari kerja vs akhir pekan:')
print(profil.group_by('jenis_hari').agg(pl.col('kecepatan_rata').min().alias('minimum'), pl.col('kecepatan_rata').mean().round(2).alias('rata_rata')))

print()
print('Korelasi kecepatan sekarang dengan horizon:')
print(korelasi.with_columns(pl.Series('horizon', nama_h)).select('horizon', 'x_t+0_d0'))

print()
print('Kecepatan rata-rata harian: awal vs akhir periode (rolling 7 hari):')
print(harian.drop_nulls('ma_7').select(pl.col('ma_7').first().round(2).alias('awal'), pl.col('ma_7').last().round(2).alias('akhir'),
                                       pl.col('kecepatan_harian').min().round(2).alias('hari_terendah')))
print(harian.sort('kecepatan_harian').head(3).select('tanggal', 'kecepatan_harian'))

print()
print('Sebaran antar sensor:')
print(per_sensor.select(
    pl.col('kecepatan_rata').min().alias('min'), pl.col('kecepatan_rata').max().alias('maks'),
    pl.col('persen_macet').min().alias('macet_min'), pl.col('persen_macet').max().alias('macet_maks'),
    pl.col('persen_macet').median().alias('macet_median'),
))

Kecepatan rata-rata jaringan: 62.62 mph

Jam dengan kecepatan rata-rata terendah dan tertinggi:
shape: (3, 2)
┌─────┬────────────────┐
│ jam ┆ kecepatan_rata │
│ --- ┆ ---            │
│ i8  ┆ f64            │
╞═════╪════════════════╡
│ 17  ┆ 54.2           │
│ 18  ┆ 56.41          │
│ 16  ┆ 56.72          │
└─────┴────────────────┘
shape: (3, 2)
┌─────┬────────────────┐
│ jam ┆ kecepatan_rata │
│ --- ┆ ---            │
│ i8  ┆ f64            │
╞═════╪════════════════╡
│ 23  ┆ 66.66          │
│ 22  ┆ 66.69          │
│ 5   ┆ 66.82          │
└─────┴────────────────┘

Hari kerja vs akhir pekan:
shape: (2, 3)
┌─────────────┬─────────┬───────────┐
│ jenis_hari  ┆ minimum ┆ rata_rata │
│ ---         ┆ ---     ┆ ---       │
│ str         ┆ f64     ┆ f64       │
╞═════════════╪═════════╪═══════════╡
│ Akhir pekan ┆ 63.19   ┆ 65.28     │
│ Hari kerja  ┆ 49.12   ┆ 60.89     │
└─────────────┴─────────┴───────────┘

Korelasi kecepatan sekarang dengan horizon:
shape: (6, 2)
┌───────────┬────────

## Bagian 4: Rumusan Insight Analitik Utama

Cell Angka Pendukung Insight dan tabel di bawah tiap visualisasi mencetak angka-angka berikut.

1. Puncak kemacetan terjadi pada sore hari, lebih parah daripada pagi (Visualisasi 4 dan 5). Kecepatan rata-rata terendah ada pada pukul 17:00 (54,2 mph), sedangkan tertinggi pukul 05:00 (66,8 mph). Porsi pengamatan macet (< 40 mph) mencapai 21,6% pada pukul 17:00 dan 16,2% pada pukul 18:00, dibanding 15,4% pada pukul 08:00. Implikasi: pengaturan lalu lintas atau peringatan dini sebaiknya diprioritaskan pada jam 16:00 s.d. 18:00.
2. Hari kerja dan akhir pekan berbeda tajam (Visualisasi 5 dan 7). Rata-rata kecepatan hari kerja 60,9 mph melawan 65,3 mph pada akhir pekan, dan titik terendah profil harian 49,1 mph (hari kerja) melawan 63,2 mph (akhir pekan), sehingga akhir pekan hampir tidak mengalami kemacetan. Kombinasi hari-jam terburuk adalah Kamis 17:00 (49,0 mph), diikuti Rabu 17:00 (49,2 mph) dan Selasa 17:00 (49,9 mph). Implikasi: model peramalan wajib memasukkan fitur hari dalam seminggu dan jam, bukan hanya nilai kecepatan sebelumnya.
3. Kemacetan terkonsentrasi pada sebagian kecil sensor (Visualisasi 6 dan 8). Persentase waktu macet per sensor berkisar 0% sampai 18,4% dengan median hanya 3,9%, dan kecepatan rata-rata antar sensor 53,9 s.d. 68,0 mph. Empat dari lima sensor tersering macet (400172, 400586, 400895, 400971) berada pada lintang 37,37 s.d. 37,38 dan bujur -121,90 s.d. -121,96, jadi saling berdekatan. Implikasi: titik rawan bersifat spasial berkelompok, sehingga model berbasis graf (hubungan antar sensor) layak dipertimbangkan.
4. Tidak ada tren jangka panjang yang kuat, tetapi ada hari anomali (Visualisasi 1 dan 9). Rata-rata bulanan stabil pada 62,2 s.d. 63,3 mph (tertinggi April 63,3, terendah Juni 62,2), dan rata-rata bergerak 7 hari bergeser dari 63,4 ke 62,3 mph. Hari terburuk adalah 18 Januari (58,3 mph), 10 Januari (58,5 mph), dan 1 Maret (58,7 mph). Implikasi: pergeseran level antar split `train`, `val`, `test` kecil, tetapi hari anomali perlu diperiksa penyebabnya (misalnya cuaca atau insiden) sebelum dianggap noise.
5. Kecepatan saat ini sangat memprediksi kecepatan dekat, namun melemah terhadap horizon (Visualisasi 3). Korelasi dengan kecepatan sekarang adalah 0,984 pada +5 menit, 0,940 pada +15 menit, 0,867 pada +30 menit, dan 0,725 pada +60 menit. Implikasi: baseline sederhana "kecepatan sekarang = kecepatan nanti" sudah sangat kuat untuk horizon pendek, sehingga model baru hanya bermakna jika unggul pada horizon 30 s.d. 60 menit.
6. Distribusi kecepatan condong ke kiri; kemacetan adalah ekor kiri (Visualisasi 2). Median 65,3 mph lebih tinggi daripada rata-rata 62,6 mph (simpangan baku 9,6), dan 9,3% pengamatan berada di bawah batas bawah IQR sensor masing-masing (Milestone 2). Implikasi: metrik berbasis rata-rata (seperti MSE) akan didominasi oleh periode macet yang jarang tetapi ekstrem, sehingga evaluasi perlu dipisah antara kondisi lancar dan macet.

### Keterbatasan Analisis

1. Celah waktu 12 Maret 2017 03:00 (selisih 1 jam 5 menit) kemungkinan artefak pergantian waktu musim panas AS dan bukan data hilang. `shift()` dan `rolling_mean()` pada data harian tidak terpengaruh, tetapi pada data 5 menit satu langkah di titik itu melompat 65 menit.
2. Periode data hanya 6 bulan (Januari sampai Juni 2017). Pola musiman tahunan tidak dapat dinilai, dan hanya siklus harian serta mingguan yang terlihat.
3. Batas kondisi lalu lintas (macet di bawah 40 mph, padat 40 sampai 60 mph) adalah kategori deskriptif buatan sendiri dan bukan standar resmi.
4. Visualisasi 3 memakai 20% sensor (`node_id % 5 == 0`) dan bukan seluruh sensor.
5. Dataset berasal dari California dan bukan dataset Indonesia, sehingga tidak memenuhi syarat "Dataset Indonesia" pada spesifikasi tugas di README. Keterbatasan ini dicatat secara eksplisit sebagai kejujuran akademik.